# RCIS Scheduler
- Uses a FCFS (First Come, First Serve) Algorithm to determine interviewee bookings for companies

## Algo Approach
- For each day, get companies and interviewees available
- For each company in that day, get all timeslots where company is available.
- Filter interviewee list (form responses) for each timeslot + company preferred degree program, order based on response time
- Get interviewee at the top, remove interviewee from list


## Dependency Installation

## Imports

In [4]:
from datetime import datetime
from typing import NewType, Protocol
from enum import Enum
import pandas as pd

## Classes and Protocols for Scheduler

In [4]:
class DegreeProgram(str, Enum):
    ChemicalEngg = 'BS Chemical Engineering'
    CivilEngg = 'BS Civil Engineering'
    ComputerEngg = 'BS Computer Engineering'
    ComputerScience = 'BS Computer Science'
    ElectricalEngg = 'BS Electrical Engineering'
    ElectronicsEngg = 'BS Electronics Engineering'
    GeodeticEngg = 'BS Geodetic Engineering'
    IndustrialEngg = 'BS Industrial Engineering'
    MaterialsEngg = 'BS Materials Engineering'
    MechEngg = 'BS Mechanical Engineering'
    MetalEngg = 'BS Metallurgical Engineering'
    MiningEngg = 'BS Mining Engineering'

class Times(str, Enum): # not ideal, but I don't really want to deal with datetime right now
# Resume Consultations
    Time0915_0945 = "09:00 AM - 09:45 AM"
    Time1000_1045 = "10:00 AM - 10:45 AM"
    Time1100_1145 = "11:00 AM - 11:45 AM"
    Time1315_1400 = "01:15 PM - 02:00 PM"
    Time1415_1500 = "02:15 PM - 03:00 PM"
    Time1515_1600 = "03:15 PM - 04:00 PM"
    Time1615_1700 = "04:15 PM - 05:00 PM"

# Interview Simulations
    Time0900_0930 = "09:00 AM - 09:30 AM"
    Time0930_1000 = "09:30 AM - 10:00 AM"
    Time1015_1045 = "10:15 AM - 10:45 AM"
    Time1045_1115 = "10:45 AM - 11:15 AM"
    Time1130_1200 = "11:30 AM - 12:00 PM"
    Time1330_1400 = "01:30 PM - 02:00 PM"
    Time1400_1430 = "02:00 PM - 02:30 PM"
    Time1445_1515 = "02:45 PM - 03:15 PM"
    Time1515_1545 = "03:15 PM - 03:45 PM"
    Time1600_1630 = "04:00 PM - 04:30 PM"
    Time1645_1715 = "04:30 PM - 05:00 PM"
class Participant(Protocol):
    @property
    def name(self) -> str:
        # Return str that is the name of the participant (could be interviewee/interviewer)
        ...
    @property
    def time_slots(self) -> list[Times]:
        # Returns time slots of participant
        ...
class Interviewee:
    def __init__(self, name: str, email: str, capes_id: str, time_slots: list[Times], registration_order: int, degree_program: DegreeProgram):
        self._name = name
        self._email = email
        self._capes_id = capes_id
        self._time_slots = time_slots
        self._degree_program = degree_program
        self._priority = len(time_slots)
        self._registration_order = registration_order
        self._granted_slot: Times
    def __eq__(self, b):
        return type(self) == type(b) and self.name == b.name 
    def __hash__(self):
        return hash(self.name)
    @property
    def name(self) -> str:
        return self._name
    @property
    def email(self) -> str:
        return self._email
    @property
    def capes_id(self) -> str:
        return self._capes_id
    @property
    def time_slots(self) -> list[Times]:
        return self._time_slots
    @property
    def degree_program(self) -> DegreeProgram:
        return self._degree_program
    @property
    def priority(self) -> int:
        return self._priority
    @property
    def registration_order(self) -> int:
        return self._registration_order
    @property
    def granted_slot(self) -> Times:
        return self._granted_slot
    def set_granted_slot(self, time: Times):
        self._granted_slot = time
class Interviewer:
    def __init__(self, name: str, time_slots: list[Times], degree_program_preference: list[DegreeProgram] | DegreeProgram, appointment_type : str):
        self._name = name
        self._time_slots = time_slots
        self._degree_program_preference = degree_program_preference
        self._interviewee_list : dict[Times, list[Interviewee]] = dict()
        self._appointment_type = appointment_type
    def __lt__(a : Interviewer, b : Interviewer):
        return a.priority < b.priority
    def unallocated_type(self, type):
        self._unallocated_type = type
    @property
    def name(self) -> str:
        return self._name
    @property
    def time_slots(self) -> list[Times]:
        return self._time_slots
    @property
    def degree_program_preference(self) -> list[DegreeProgram] | DegreeProgram:
        return self._degree_program_preference
    @property
    def interviewee_list(self) -> dict[Times, list[Interviewee]]:
        return self._interviewee_list
    @property
    def priority(self):
        return len(self._time_slots)
    @property
    def appointment_type(self):
        return self._appointment_type
    def add_to_interviewee_list(self, interviewee: Interviewee, time : Times, max_alloc: int):
        if time not in self._interviewee_list.keys():
            self._interviewee_list[time] = []
        elif len(self._interviewee_list[time]) >= max_alloc:
            return 0
        interviewee.set_granted_slot(time)
        self._interviewee_list[time].append(interviewee)
        return 1    
    

## Get Interviewe/Response Data

In [37]:
responses_url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vR5hzOahI90d182BRnnk_lF68X3GCWfbr6rpAIeds1ezrABvujUhMS9JN_maoTAtiJ11qx2eCf6SyIL/pub?gid=1183642779&single=true&output=csv"

interviewee_df = pd.read_csv(responses_url)

interviewee_df.head()

,Timestamp,Data Privacy Agreement,CAPES ID,Are you representing any UPD College of Engineering Organization in joining this event?,Updated Resume or CV,What sub-event would you like to register for?,What is your most preferred date? [Resume Consultation (RC)],What is your most preferred date? [Interview Simulation (IS)],"What are your preferred time slots for RC, November 13 (Thursday)? [9:00-9:30 AM]","What are your preferred time slots for RC, November 13 (Thursday)? [9:30-10:00 AM]",...,"What are your preferred time slots for IS, November 13 (Thursday)? [2:15-3:00 PM]","What are your preferred time slots for IS, November 13 (Thursday)? [3:15-4:00 PM]","What are your preferred time slots for IS, November 13 (Thursday)? [4:15-5:00 PM]","What are your preferred time slots for IS, November 14 (Friday)? [9:00-9:45 AM]","What are your preferred time slots for IS, November 14 (Friday)? [10:00-10:45 AM]","What are your preferred time slots for IS, November 14 (Friday)? [11:00-11:45 AM]","What are your preferred time slots for IS, November 14 (Friday)? [1:15-2:00 PM]","What are your preferred time slots for IS, November 14 (Friday)? [2:15-3:00 PM]","What are your preferred time slots for IS, November 14 (Friday)? [3:15-4:00 PM]","What are your preferred time slots for IS, November 14 (Friday)? [4:15-5:00 PM]"
0,11/2/2025 20:03:04,I agree,xmjaudalso,No,https://drive.google.com/open?id=1MqWge49TDvC0...,Resume Consultations (RC) ONLY,November 13 (Thu),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11/4/2025 16:38:15,I agree,cecarpio,No,https://drive.google.com/open?id=1J7RqQFHnOZMs...,Both (RC & IS),November 14 (Fri),November 14 (Fri),NaN,NaN,...,NaN,NaN,NaN,7.0,6.0,5.0,4.0,1.0,2.0,3.0
2,11/6/2025 0:57:43,I agree,mdpunzalan,No,https://drive.google.com/open?id=1gkdZswnlbqPg...,Both (RC & IS),November 13 (Thu),November 13 (Thu),NaN,NaN,...,NaN,1.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
3,11/6/2025 12:33:38,I agree,mmurillo,No,https://drive.google.com/open?id=18hwpPcI2noW1...,Both (RC & IS),November 14 (Fri),November 14 (Fri),NaN,NaN,...,NaN,NaN,NaN,7.0,6.0,1.0,2.0,3.0,4.0,5.0
4,11/6/2025 14:42:04,I agree,cmborrega,No,https://drive.google.com/open?id=1eND6TnRl__uo...,Both (RC & IS),November 13 (Thu),November 13 (Thu),NaN,NaN,...,NaN,NaN,NaN,1.0,2.0,3.0,4.0,NaN,NaN,NaN


## Data Cleaning

### Column Selection for Cleaning

In [38]:
# RC/IS date column name
date_column_filter = 'What is your most preferred date?'



In [42]:
# Get columns to clean
int_cols = interviewee_df.select_dtypes(include='float').columns
rcis_date_cols = interviewee_df.columns[interviewee_df.columns.str.contains(date_column_filter)]

# Remove NaN values
interviewee_df[int_cols] = interviewee_df[int_cols].fillna(12).astype('int') 
interviewee_df[rcis_date_cols] = interviewee_df[rcis_date_cols].fillna('None')

## Scheduler

- Sort interviewees by increasing rank and timestamp for a certain day and timeslot
- Get a company
- Filter out interviewee list based on company requirements (course, etc.)
- FCFS picking
- Remove chosen interviewees from list

In [ ]:
rc_date_filter = ['RC, November 13', 'RC, November 14']
is_date_filter = ['IS, November 13', 'IS, November 14']

columns = interviewee_df.columns[interviewee_df.columns.str.contains(is_date_filter[0])]

interviewee_df = interviewee_df.sort_values(by=[columns[-2],'Timestamp'], ascending=[True, True])


What are your preferred time slots for IS, November 13 (Thursday)? [3:15-4:00 PM]


,Timestamp,Data Privacy Agreement,CAPES ID,Are you representing any UPD College of Engineering Organization in joining this event?,Updated Resume or CV,What sub-event would you like to register for?,What is your most preferred date? [Resume Consultation (RC)],What is your most preferred date? [Interview Simulation (IS)],"What are your preferred time slots for RC, November 13 (Thursday)? [9:00-9:30 AM]","What are your preferred time slots for RC, November 13 (Thursday)? [9:30-10:00 AM]",...,"What are your preferred time slots for IS, November 13 (Thursday)? [2:15-3:00 PM]","What are your preferred time slots for IS, November 13 (Thursday)? [3:15-4:00 PM]","What are your preferred time slots for IS, November 13 (Thursday)? [4:15-5:00 PM]","What are your preferred time slots for IS, November 14 (Friday)? [9:00-9:45 AM]","What are your preferred time slots for IS, November 14 (Friday)? [10:00-10:45 AM]","What are your preferred time slots for IS, November 14 (Friday)? [11:00-11:45 AM]","What are your preferred time slots for IS, November 14 (Friday)? [1:15-2:00 PM]","What are your preferred time slots for IS, November 14 (Friday)? [2:15-3:00 PM]","What are your preferred time slots for IS, November 14 (Friday)? [3:15-4:00 PM]","What are your preferred time slots for IS, November 14 (Friday)? [4:15-5:00 PM]"
2,11/6/2025 0:57:43,I agree,mdpunzalan,No,https://drive.google.com/open?id=1gkdZswnlbqPg...,Both (RC & IS),November 13 (Thu),November 13 (Thu),12,12,...,12,1,2,1,12,12,12,12,12,12
5,11/6/2025 16:08:45,I agree,lchernandez,No,https://drive.google.com/open?id=1r2ipJRzMnMrg...,Resume Consultations (RC) ONLY,November 13 (Thu),None,9,8,...,1,2,7,1,2,3,4,5,6,7
6,11/6/2025 17:08:06,I agree,jglao,No,https://drive.google.com/open?id=1F-eUumjv0JR0...,Both (RC & IS),November 14 (Fri),November 14 (Fri),11,10,...,5,3,4,7,6,5,1,3,4,2
0,11/2/2025 20:03:04,I agree,xmjaudalso,No,https://drive.google.com/open?id=1MqWge49TDvC0...,Resume Consultations (RC) ONLY,November 13 (Thu),None,12,12,...,12,12,12,12,12,12,12,12,12,12
1,11/4/2025 16:38:15,I agree,cecarpio,No,https://drive.google.com/open?id=1J7RqQFHnOZMs...,Both (RC & IS),November 14 (Fri),November 14 (Fri),12,12,...,12,12,12,7,6,5,4,1,2,3
